## Overfitting and Underfitting Analysis
## 过拟合与欠拟合分析

### 1. Motivation / 分析动机

The primary goal of this project is to examine whether incorporating neighbourhood-based statistical features can improve house price prediction performance.

本项目的核心目标是评估**加入邻域统计特征是否能够提升房价预测模型的性能**。

However, evaluating model quality using **Test Mean Squared Error (MSE) alone is insufficient**.  
A lower Test MSE may arise from two different causes:

1. genuine improvement in model generalisation, or  
2. overfitting, where the model memorises training data but fails to generalise.

然而，仅通过**测试集 MSE**来评估模型质量是不充分的。  
测试误差的降低可能来自两种不同情况：

1. 模型泛化能力的真实提升  
2. 模型对训练数据的过度拟合（过拟合）

To distinguish between these possibilities, we compare **Train MSE and Test MSE** for each model.  
The relative difference between these two metrics provides an indication of the model's generalisation behaviour.

因此，本分析通过对比**训练集 MSE 与测试集 MSE**来判断模型的泛化表现。  
两者之间的相对差距可以帮助识别潜在的过拟合或欠拟合问题。


### 2. Diagnostic Method / 诊断方法

We define the **Test-to-Train MSE Ratio** as a simple diagnostic indicator:

我们定义 **测试集 / 训练集 MSE 比值** 作为主要诊断指标：

Ratio = Test MSE / Train MSE

Diagnosis is based on **relative comparison rather than absolute error thresholds**,  
since the absolute magnitude of MSE depends on the scale of the dataset.

诊断基于**相对误差比较**而非绝对阈值，因为 MSE 的数值大小取决于数据集的尺度。

| Condition / 条件 | Interpretation / 解释 |
|---|---|
| Ratio > 1.5 | Possible overfitting — test error substantially larger than training error |
| Ratio < 1.1 | Small generalisation gap — if both errors remain high, this may indicate underfitting |
| Otherwise | Reasonable fit — no large train–test gap observed |

The value **1.5** is used as a heuristic diagnostic threshold.  
If the test error exceeds the training error by more than approximately 50%, the model may exhibit signs of overfitting.

阈值 **1.5** 被用作经验性的诊断界限。  
若测试误差比训练误差高出超过约 **50%**，则模型可能存在过拟合风险。

In [ ]:
# ============================================================
# Overfitting / Underfitting Check
# 过拟合 / 欠拟合检查
#
# Motivation / 目的:
#   Reporting Test MSE alone is insufficient to evaluate model quality.
#   A lower Test MSE could result from genuine generalisation improvement,
#   or simply from the model overfitting the training data.
#   By comparing Train MSE and Test MSE, we can assess whether the
#   improvement in Model 2 is more likely to reflect better generalisation
#   rather than a larger train–test gap.

#   仅汇报测试集MSE不足以评估模型质量。
#   更低的测试集MSE可能来自真实的泛化能力提升，
#   也可能只是模型过拟合训练数据的结果。
#   通过对比训练集MSE和测试集MSE，我们可以辅助判断模型2的提升
#   是否更可能来自更好的泛化能力，而不是更大的训练-测试误差差距。

# Why Train vs Test (not Train vs Val) / 为什么用训练集 vs 测试集（而非验证集）:
#   In the final training stage, the training and validation sets are
#   merged together to maximise the amount of data available for training.
#   This means no independent validation set remains.
#   The held-out test set therefore serves as the reference for
#   evaluating generalisation performance.
#
#   在最终训练阶段，训练集和验证集被合并以最大化可用训练数据。
#   因此不再有独立的验证集可用。
#   保留的测试集因此作为评估泛化性能的参考集。
# ============================================================
import matplotlib.pyplot as plt
import numpy as np

# ── Diagnosis helper / 诊断辅助函数 ──────────────────────────
def diagnose_final(model_name, train_mse, test_mse, threshold_ratio=1.5):  # heuristic threshold / 经验阈值
    """
    Print an overfitting/underfitting diagnosis based on Test/Train MSE ratio.
    根据测试集与训练集MSE比值，打印过拟合/欠拟合诊断结果。

    Diagnosis rules / 诊断规则:
        ratio > 1.5                      → Possible overfitting
        比值 > 1.5                        → 可能存在过拟合：测试集误差明显高于训练集
        ratio < 1.1                      → Low generalisation gap; possible underfitting if both errors remain high
        比值 < 1.1                        → 泛化差距较小；若两者均较高则可能欠拟合
        Otherwise                        → Good fit
        否则                              → 拟合良好
    """
    # Compute ratio of test MSE to train MSE
    # If train_mse is zero, set ratio to infinity to avoid division by zero
    # 计算测试集MSE与训练集MSE的比值
    # 若训练集MSE为零则设为无穷大，避免除零错误
    ratio = test_mse / train_mse if train_mse > 0 else float('inf')

    print(f"\n{'='*55}")
    print(f"  {model_name}")
    print(f"{'='*55}")
    print(f"  Train MSE       : {train_mse:,.4f}")
    print(f"  Test  MSE       : {test_mse:,.4f}")
    print(f"  Test/Train Ratio: {ratio:.3f}")

    # Diagnosis is based on relative comparison between train and test MSE,
    # rather than absolute thresholds, which are dataset-dependent.
    # 诊断基于训练集和测试集MSE的相对比较，而非绝对阈值，
    # 因为绝对阈值依赖于具体数据集的MSE量级。
    if ratio > threshold_ratio:
        # Test MSE is much larger than Train MSE
        # → model memorised training data but fails to generalise → overfitting
        # 测试集MSE远大于训练集MSE
        # → 模型过度记忆训练数据，泛化能力差 → 过拟合
        print("  ⚠ Diagnosis : POSSIBLE OVERFITTING — test MSE is notably larger than train MSE")
        print("  ⚠ 诊断结果  : 可能存在过拟合 — 测试集误差明显高于训练集误差")
    elif ratio < 1.1:
        # Train and test MSE are similar (ratio < 1.1)
        # If both values remain relatively high, this may indicate underfitting
        # — the model lacks sufficient capacity to learn the underlying pattern.
        # No absolute threshold is used here as MSE scale is dataset-dependent.
        # 训练集与测试集MSE相近（比值 < 1.1）
        # 若两者均处于相对较高的水平，则可能存在欠拟合
        # — 模型容量不足，无法学习数据中的潜在规律。
        # 此处不使用绝对阈值，因为MSE的量级取决于具体数据集。
        print("  ~ Diagnosis : LOW GENERALISATION GAP — train and test errors are similar")
        print("  ~ 诊断结果  : 泛化差距较小 — 训练集与测试集误差接近")
    else:
        # Train and test MSE are close and both reasonably low
        # → model generalises well → good fit
        # 训练集与测试集误差接近且均处于合理水平
        # → 模型泛化能力良好 → 拟合良好
        print("  ✓ Diagnosis : REASONABLE FIT — no large train–test gap is observed")
        print("  ✓ 诊断结果  : 拟合较合理 — 未观察到明显的训练测试误差差距")
    print(f"{'='*55}")


### 3. Results / 实验结果

| Model / 模型 | Train MSE | Test MSE | Ratio | Diagnosis |
|---|---|---|---|---|
| Model 1 — Baseline | 0.2398 | 0.2485 | 1.036 | ✓ Reasonable Fit |
| Model 2 — Enhanced | 0.1557 | 0.1805 | 1.159 | ✓ Reasonable Fit |

Both models exhibit **relatively small train–test gaps**, suggesting that neither model shows strong evidence of overfitting or severe underfitting.

两个模型的**训练误差与测试误差差距均较小**，说明未观察到明显的过拟合或严重欠拟合现象。

Model 2 achieves a **lower Test MSE** than Model 1, suggesting that the addition of neighbourhood-based features improves predictive performance.

模型 2 的**测试误差明显低于模型 1**，说明加入邻域统计特征后模型预测性能有所提升。

Although Model 2 shows a slightly larger Test/Train ratio, the value remains well below the overfitting threshold, indicating that the improvement is unlikely to be caused by excessive memorisation of the training data.

尽管模型 2 的 Test/Train 比值略高，但仍远低于过拟合阈值，说明该性能提升**不太可能来自训练数据记忆**。

Overall, the results suggest that neighbourhood-based features contribute to improved generalisation performance without introducing clear signs of overfitting.

总体而言，结果表明邻域特征有助于提升模型的泛化能力，并未引入明显的过拟合问题。

In [ ]:
# ============================================================
# MODEL 1 — Final Baseline
# 模型1 — 最终基线模型
#
# baseline_final_model was trained on the merged train+val set
# using only the original features, with no neighbourhood features.
# Both baseline_train_mse and baseline_test_mse_final were already
# computed immediately after baseline_final_model was trained,
# so we reuse them directly here without any additional computation.
#
# baseline_final_model 在合并后的 train+val 集上训练，仅使用原始特征。
# baseline_train_mse 和 baseline_test_mse_final 在
# baseline_final_model 训练完成后已经立即计算好，直接复用。
# ============================================================

# Reuse already-computed values — no re-training needed
# 直接复用已计算好的值，无需重新训练
m1_train_mse = baseline_train_mse        # already computed above / 原代码已算好
m1_test_mse  = baseline_test_mse_final   # already computed above / 原代码已算好

diagnose_final("Model 1 — Final Baseline", m1_train_mse, m1_test_mse)


# ============================================================
# MODEL 2 — Final Enhanced
# 模型2 — 最终增强模型（含邻域统计特征）
#
# model_enh_final was trained on the merged train+val set
# with neighbourhood-based price statistics added as extra inputs.
# enhanced_test_mse_final was already computed immediately after
# model_enh_final was trained, so we reuse it directly.
#
# However, Train MSE for Model 2 was never computed in the original
# code, so we compute it once here using the already-fitted model.
# No re-training is required.
#
# model_enh_final 在合并后的 train+val 集上训练，包含邻域统计特征。
# enhanced_test_mse_final 已在训练后立即计算，直接复用。
# 但模型2的训练集MSE在原代码中从未计算过，
# 因此在此处使用已训练好的模型计算一次，无需重新训练。
# ============================================================

# Reuse already-computed test MSE — no re-training needed
# 直接复用已计算好的测试集MSE，无需重新训练
m2_test_mse  = enhanced_test_mse_final   # already computed above / 原代码已算好

# Train MSE for Model 2 was not computed in the original code
# Compute it once here using the already-fitted model_enh_final
# 模型2的训练集MSE在原代码中未计算，此处用已训练好的模型补充计算一次
m2_train_mse = mean_squared_error(        # not in original code, compute once here
    y_train_final,                        # 原代码未计算，在此计算一次
    model_enh_final.predict(X_train_enh_final)
)

diagnose_final("Model 2 — Final Enhanced", m2_train_mse, m2_test_mse)

### Figure Caption / 图注说明

**Figure 1 — Train vs Test MSE Comparison**

Bar chart comparing the training and test mean squared errors of the two final models.  
The gap between Train MSE and Test MSE provides a visual indication of potential overfitting.  
A small gap suggests that the model generalises well beyond the training data.

图 1 展示了两个最终模型的训练集和测试集 MSE 对比。  
训练误差与测试误差之间的差距可以直观反映潜在的过拟合情况。  
较小的误差差距通常意味着模型具有较好的泛化能力。

**Figure 2 — Approximate Learning Curves**

Approximate learning curves showing the evolution of training MSE across optimisation iterations.  
The solid line represents the training error trajectory, while the horizontal dashed line  
indicates the final test error as a reference.

These curves provide a qualitative view of the optimisation behaviour of the models,  
but they should not be interpreted as exact epoch-level training logs.

图 2 展示了训练过程中训练误差的近似变化趋势。  
实线表示训练误差曲线，水平虚线表示最终测试误差作为参考。

该图主要用于提供模型优化过程的定性观察，  
而不应被理解为精确的逐 epoch 训练记录。


### 4. Learning Curve Analysis / 学习曲线分析

In addition to the final MSE comparison, **approximate learning curves** are plotted to visualise how the training error evolves during optimisation.

除了最终误差比较外，本研究还绘制了**近似学习曲线**，用于观察训练误差在训练过程中的变化趋势。

The curves are generated by **re-simulating training from scratch** using `warm_start=True` with `max_iter=1` in each call to `fit()`.

这些曲线通过使用 `warm_start=True` 且每次 `fit()` 设置 `max_iter=1` 从头重新模拟训练过程得到。

Although the architecture, hyperparameters, and random seed are identical to the final models, this procedure **does not reproduce the exact training history** of the fitted models.

尽管模型架构、超参数和随机种子与最终模型一致，该过程**并不能完全复现真实训练轨迹**。

In sklearn's `MLPRegressor`, each call to `fit(max_iter=1)` does **not strictly correspond to a single epoch**, because the optimiser maintains internal states and the training data may be shuffled.

在 sklearn 的 `MLPRegressor` 中，每次 `fit(max_iter=1)` **并不严格等价于一个 epoch**，因为优化器具有内部状态并且训练数据可能被重新打乱。

Therefore, the resulting curves should be interpreted as **approximate optimisation trajectories** rather than exact epoch-level training logs.

因此，这些曲线应被理解为**近似的优化轨迹**，而非精确的逐 epoch 训练记录。

Only the **training error** is plotted per iteration, while the final **test error** is shown as a horizontal reference line.

图中仅绘制**训练误差曲线**，最终测试误差仅作为一条水平参考线展示。

This design avoids evaluating the test set repeatedly during training, which would introduce methodological bias.

这种设计避免在训练过程中反复使用测试集，从而防止方法学上的偏差。

In [ ]:
# ============================================================
# VISUALISATION 1 — Bar chart: Train vs Test MSE
# 可视化1 — 柱状图：训练集与测试集MSE对比
#
# Purpose / 目的:
#   Visually compare the gap between Train MSE and Test MSE for both
#   models. A larger gap may indicate possible overfitting.
#   直观对比两个模型训练集和测试集的MSE差距。
#   更大的差距可能提示存在过拟合。
# ============================================================

labels     = ['Model 1\n(Baseline)', 'Model 2\n(Enhanced)']
train_mses = [m1_train_mse, m2_train_mse]
test_mses  = [m1_test_mse,  m2_test_mse]

# X-axis positions for each group of bars / 每组柱子的x轴位置
x     = np.arange(len(labels))
width = 0.35  # Width of each individual bar / 每个柱子的宽度

fig, ax = plt.subplots(figsize=(8, 5))

# Plot Train MSE bars on the left of each group
# 在每组左侧绘制训练集MSE柱子
bars1 = ax.bar(x - width/2, train_mses, width, label='Train MSE', color='steelblue',  alpha=0.85)

# Plot Test MSE bars on the right of each group
# 在每组右侧绘制测试集MSE柱子
bars2 = ax.bar(x + width/2, test_mses,  width, label='Test MSE',  color='darkorange', alpha=0.85)

ax.set_ylabel('MSE')
ax.set_title('Train vs Test MSE — Model 1 and Model 2')
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.legend()

# Annotate each bar with its exact MSE value
# We use ax.text() instead of bar_label() for compatibility
# with older versions of matplotlib
# 在每个柱子顶部标注具体MSE数值
# 使用 ax.text() 而非 bar_label()，以兼容旧版本的 matplotlib
for bar, val in zip(bars1, train_mses):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
            f'{val:.4f}', ha='center', va='bottom', fontsize=9)
for bar, val in zip(bars2, test_mses):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
            f'{val:.4f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()


# ============================================================
# VISUALISATION 2 — Approximate Learning Curves (Train MSE vs Epoch)
# 可视化2 — 近似学习曲线（训练集MSE随训练轮数变化）
#
# Purpose / 目的:
#   The ratio test above evaluates only the final model state.
#   To provide additional insight, approximate learning curves
#   are plotted to illustrate how Train MSE evolves during training
#   under the same architecture and hyperparameters as the final models.
#
#   上述比值检验仅反映模型最终状态。
#   为进一步观察训练过程，这里绘制近似学习曲线，
#   展示在相同模型结构和超参数下训练误差的变化趋势。
#
# Methodological note / 方法说明:
#   Training is re-simulated using warm_start=True with max_iter=1
#   per fit() call. This approximates the optimisation trajectory
#   but does NOT reproduce the exact training history of the final model.
#
#   在 sklearn 的 MLPRegressor 中，由于 Adam 优化器内部状态和
#   数据随机打乱，每次 fit() 调用并不严格等价于一个 epoch，
#   因此该曲线应被理解为近似训练趋势，而非精确逐轮记录。
# ============================================================


def collect_train_curve(Xtr, ytr, hidden, act, max_iter=800):
    """
    Simulate training with warm_start and record Train MSE at each step.
    使用 warm_start 逐步训练并记录每一步的训练集MSE。

    This produces an approximate learning curve rather than the
    exact optimisation path of the final fitted model.
    该曲线为近似训练趋势，而非最终模型真实训练历史。

    The test set is NOT evaluated per epoch to avoid using it
    as a model selection reference.
    测试集不会参与逐轮评估，以避免用于模型选择。

    Returns:
        list: training MSE values recorded after each step
        每一步记录的训练集MSE列表
    """
    # warm_start=True retains weights between calls so training is continuous
    # warm_start=True 保留上次权重，使训练连续进行，不重新初始化
    m = MLPRegressor(
        hidden_layer_sizes=hidden,
        activation=act,
        solver='adam',
        learning_rate_init=1e-3,
        max_iter=1,        # Each call performs a short training step / 每次调用执行一次短训练
        warm_start=True,   # Continue from previous weights / 保留权重继续训练
        early_stopping=False,
        random_state=26
    )

# Design choice / 设计说明:
#   Only Train MSE is plotted per epoch. The Test MSE is shown as a
#   single horizontal reference line to avoid using the test set
#   for model selection during training.
#
#   每个 epoch 仅绘制训练集MSE曲线，测试集MSE仅以水平参考线展示，
#   以避免在训练过程中将测试集用于模型选择。

    train_curve = []
    for _ in range(max_iter):
        m.fit(Xtr, ytr)  # Train for one more epoch / 再训练一个epoch

        # Record Train MSE after this epoch
        # 记录本epoch结束后的训练集MSE
        pred = m.predict(Xtr)
        train_curve.append(mean_squared_error(ytr, pred))
    return train_curve

### 5. Interpreting Learning Curves / 学习曲线解读

| Pattern / 曲线形态 | Interpretation / 解读 |
|---|---|
| Train curve converges near test reference line | May indicate good generalisation |
| Train curve falls far below test reference line | May indicate potential overfitting |
| Both train curve and test line remain high | May indicate possible underfitting |

These interpretations should be treated as **qualitative indicators** rather than definitive diagnostic criteria.

上述解读应被视为**定性参考**，而非严格诊断结论。

This is because no independent validation trajectory is available and the learning curves themselves are approximations.

原因在于没有独立验证集曲线，且学习曲线本身为近似结果。

In [ ]:
#  Interpretation / 曲线解读:
#   Train curve approaching the test reference → potential good generalisation
#   Train curve far below the test reference → possible overfitting
#   Both errors remaining high → possible underfitting
#
#   训练曲线接近测试参考线 → 可能泛化良好
#   训练曲线明显低于测试参考线 → 可能过拟合
#   两者均较高 → 可能欠拟合
#
#   These interpretations are qualitative indicators rather than
#   definitive diagnostic criteria.
#   以上解读仅作为定性参考，而非确定性诊断标准。

print("\nCollecting learning curves (this may take a moment)…")
print("正在收集学习曲线数据（可能需要一点时间）…")

# Collect approximate train curve for Model 1 — baseline features only
# 收集模型1的近似训练曲线，仅使用基线特征
lc_train_m1 = collect_train_curve(
    X_train_final_base, y_train_final_base,
    hidden=(128, 32), act='tanh'
)

# Collect approximate train curve for Model 2 — with neighbourhood features
# 收集模型2的近似训练曲线，包含邻域统计特征
lc_train_m2 = collect_train_curve(
    X_train_enh_final, y_train_final,
    hidden=(128, 32), act='tanh'
)

# Plot approximate learning curves for both models side by side
# 并排绘制两个模型的近似学习曲线
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, lc_tr, final_test_mse, title in zip(
    axes,
    [lc_train_m1, lc_train_m2],
    [m1_test_mse,  m2_test_mse],
    ['Model 1 — Final Baseline', 'Model 2 — Final Enhanced']
):
    epochs = range(1, len(lc_tr) + 1)

    # Solid blue line = Train MSE per epoch (approximate)
    # 蓝色实线 = 每个epoch的训练集MSE（近似）
    ax.plot(epochs, lc_tr, label='Train MSE (approx.)', color='steelblue')

    # Horizontal dashed orange line = final Test MSE (reference only)
    # The test set is evaluated only once at the end, not per epoch
    # 橙色水平虚线 = 最终测试集MSE（仅作参考）
    # 测试集只在最终评估时使用一次，不参与逐epoch计算
    ax.axhline(
        y=final_test_mse,
        color='darkorange',
        linestyle='--',
        linewidth=1.2,
        label=f'Final Test MSE = {final_test_mse:.4f}'
    )

    ax.set_title(title)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('MSE')
    ax.legend(fontsize=8)
    ax.grid(True, linestyle=':', alpha=0.6)

plt.suptitle(
    'Approximate Learning Curves — Overfitting / Underfitting Check',
    fontsize=13
)
plt.tight_layout()
plt.show()


### 6. Final Discussion / 最终讨论

The analysis suggests that incorporating neighbourhood-based statistical features  
improves predictive performance while maintaining a reasonable level of generalisation.

分析结果表明，加入邻域统计特征后模型预测性能有所提升，同时仍保持良好的泛化能力。

Compared with the baseline model, the enhanced model achieves a lower test error  
without introducing a large train–test gap, which indicates that the improvement  
is unlikely to be caused by excessive memorisation of the training data.

与基线模型相比，增强模型在没有明显扩大训练—测试误差差距的情况下  
实现了更低的测试误差，这表明性能提升不太可能来自对训练数据的过度记忆。

Overall, the results provide empirical evidence that neighbourhood information  
can serve as a useful feature for improving house price prediction models.

总体而言，实验结果表明邻域信息是一类有效的特征，  
能够帮助提升房价预测模型的性能。